# Task 5: Compare all the models
We have four models now:
Naive Bayes, Logistic Regression, LSTM and BERT.
All of them were tested on the same 1,026 messages.
Here we compare them and look at the messages they got wrong.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix
nb = pd.read_csv('../data/pred_nb.csv')
lr = pd.read_csv('../data/pred_lr.csv')
lstm = pd.read_csv('../data/pred_lstm.csv')
bert = pd.read_csv('../data/pred_bert.csv')
print(nb.shape, lr.shape, lstm.shape, bert.shape)
print('Same messages:', (nb['text'] == lr['text']).all(),
      (nb['text'] == lstm['text']).all(), (nb['text'] == bert['text']).all())
y = (nb['label'] == 'spam').astype(int)
print('Spam in test:', y.sum())

(1026, 4) (1026, 4) (1026, 4) (1026, 4)
Same messages: True True True
Spam in test: 125


## 1. The four models side by side
We also add a very simple model that always says "ham",
to see what our models really add.

In [2]:
models = [('Naive Bayes', nb), ('Logistic Regression', lr),
          ('LSTM', lstm), ('BERT', bert)]
rows = []
for name, d in models:
    p = (d['pred'] == 'spam').astype(int)
    cm = confusion_matrix(y, p)
    rows.append([name,
                 round(100 * accuracy_score(y, p), 1),
                 round(100 * precision_score(y, p), 1),
                 round(100 * recall_score(y, p), 1),
                 round(100 * f1_score(y, p), 1),
                 cm[0, 1], cm[1, 0]])
rows.append(['Always say ham', round(100 * (y == 0).mean(), 1),
             0.0, 0.0, 0.0, 0, int(y.sum())])
table = pd.DataFrame(rows, columns=['model', 'accuracy', 'precision',
                                    'recall', 'f1', 'false_alarms', 'missed'])
print(table.to_string(index=False))

              model  accuracy  precision  recall   f1  false_alarms  missed
        Naive Bayes      98.9      100.0    91.2 95.4             0      11
Logistic Regression      99.1       97.5    95.2 96.4             3       6
               LSTM      98.9       95.2    96.0 95.6             6       5
               BERT      99.0       98.3    93.6 95.9             2       8
     Always say ham      87.8        0.0     0.0  0.0             0     125


## 2. Do the models agree?
We put the four answers next to each other,
and we count how many models said "spam" for each message.

In [3]:
both = nb[['label', 'text']].copy()
both['nb'] = nb['pred']
both['lr'] = lr['pred']
both['lstm'] = lstm['pred']
both['bert'] = bert['pred']
both['votes'] = (both[['nb', 'lr', 'lstm', 'bert']] == 'spam').sum(axis=1)
agree = (both['votes'] == 0) | (both['votes'] == 4)
print('All four agree on:', agree.sum(), 'messages')
print('They disagree on :', (~agree).sum(), 'messages')
print()
print('Spam missed by all four models:',
      ((both['label'] == 'spam') & (both['votes'] == 0)).sum())
print('Ham called spam by all four models:',
      ((both['label'] == 'ham') & (both['votes'] == 4)).sum())

All four agree on: 1007 messages
They disagree on : 19 messages

Spam missed by all four models: 3
Ham called spam by all four models: 0


## 3. The messages where they disagree
First, the spam messages that all four models missed.
Then, the messages where the models gave different answers.
We look at the real messages, to understand why.

In [4]:
print('=' * 15, 'SPAM missed by all four models')
for r in both[(both['label'] == 'spam') & (both['votes'] == 0)].itertuples():
    print('   ', r.text[:150])
print()
for label in ['spam', 'ham']:
    print('=' * 15, 'TRUE', label.upper(), '(the models disagree)')
    sub = both[(~agree) & (both['label'] == label)]
    for r in sub.itertuples():
        print('nb:', r.nb, '| lr:', r.lr, '| lstm:', r.lstm, '| bert:', r.bert)
        print('   ', r.text[:110])
    print()

=============== SPAM missed by all four models
    Do you ever notice that when you're driving, anyone going slower than you is an idiot and everyone driving faster than you is a maniac?
    Would you like to see my XXX pics they are so hot they were nearly banned in the uk!
    Sorry I missed your call let's talk when you have the time. I'm on 07090201529

=============== TRUE SPAM (the models disagree)
nb: ham | lr: ham | lstm: spam | bert: ham
    TBS/PERSOLVO. been chasing us since Sept for£38 definitely not paying now thanks to your information. We will 
nb: spam | lr: spam | lstm: ham | bert: ham
    RECPT 1/3. You have ordered a Ringtone. Your order is being processed...
nb: spam | lr: spam | lstm: spam | bert: ham
    Bought one ringtone and now getting texts costing 3 pound offering more tones etc
nb: ham | lr: spam | lstm: spam | bert: ham
    SMS. ac JSco: Energy is high, but u may not know where 2channel it. 2day ur leadership skills r strong. Psychi
nb: ham | lr: spam | ls

## 4. What we see
- All four models agree on 1,007 messages and disagree on 19.
- 3 spam messages were missed by all four models.
  They do not have the usual spam signs: a message about drivers,
  a message about "XXX pics", and "Sorry I missed your call...".
- BERT is the only model that got "LIFE has never been this much fun..." right.
- BERT has 2 false alarms. One is the message with the website www.fullonsms.com
  (three models called it spam). The other is a message that is only a website address.
- BERT missed 8 spam messages: the 3 above and 5 others.
  The LSTM got 4 of these 5 right.
- The LSTM called 5 normal messages spam alone, and BERT got all 5 right.
  Some of these messages are written in a strange way
  (for example "Ee msg na poortiyagi..." and "Hello-/@drivby...").
  BERT cuts a word it does not know into small pieces, so this may help it.
  This is only a sign, because we have only a few messages.
- No model is the best in everything:
  Naive Bayes has no false alarms, Logistic Regression has the best F1,
  and the LSTM misses the fewest spam messages.

## 5. Save the comparison
We save the table and all the answers in new files.
Abdullah will use them in the report and in the charts.

In [5]:
table.to_csv('../data/model_comparison_final.csv', index=False)
both.to_csv('../data/all_predictions_final.csv', index=False)
print('Saved')
print(table.to_string(index=False))

Saved
              model  accuracy  precision  recall   f1  false_alarms  missed
        Naive Bayes      98.9      100.0    91.2 95.4             0      11
Logistic Regression      99.1       97.5    95.2 96.4             3       6
               LSTM      98.9       95.2    96.0 95.6             6       5
               BERT      99.0       98.3    93.6 95.9             2       8
     Always say ham      87.8        0.0     0.0  0.0             0     125
